## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


> 확률적 경사 하강법: 전체 훈련 데이터를 로드할 필요 없이 한번에 하나의 훈련 샘플만 처리하기 때문.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

> 과적합 또는 학습률이 높은 경우 다음과 같은 일이 발생할 수 있음.

> 따라서 GridSearchCV와 같은 기술로 학습률을 조정하거나, Ridge, Lasso, Elastic Net과 같은 정규화 기법을 더 강하게 적용해 과적합을 방지할 수 있음.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

> 높은 편향을 가지는 문제. 즉, 과소적합의 문제이다. 이 문제를 해결하려면 모델의 복잡도를 높여야 하며, 이에 따라 규제 하이퍼파라미터 alpha 값을 줄여야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

1) 다중공선성 처리를 위함. 특성들 간에 강한 상관관계가 있을 때, 일반 선형 회귀는 계수가 불안정해지고 분산이 커질 수 있기 떄문. 릿지 회귀는 규제를 통해 계수들을 0에 가깝게 축소시키기 때문에 모델의 안정성을 높일 수 있음.

2) 라쏘 회귀는 1 규제를 사용하여 덜 중요한 특성의 계수를 아예 0으로 만들어 버리기 때문에 모델을 더 간결하게 만들고 해석하기 쉽게 하며, 불필요한 특성으로 인한 노이즈를 줄일 수 있기 때문.

3) 다중공선성 문제 때문. 라쏘 회귀는 상관관계가 높은 특성들 중 하나만 선택하고 나머지는 버리는 경향이 있음. 엘라스틱넷은 릿지와 라쏘 규제를 모두 결합하여 이러한 라쏘의 단점을 보완하기 때문에 더 안정적임.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [1]:
import numpy as np
from sklearn import datasets

# Iris 데이터 로드
iris = datasets.load_iris()
X = iris["data"]
y = iris["target"]

# 데이터 확인
print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("First 5 samples of X:\n", X[:5])
print("First 5 samples of y:\n", y[:5])

Features (X) shape: (150, 4)
Target (y) shape: (150,)
First 5 samples of X:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
First 5 samples of y:
 [0 0 0 0 0]


데이터 전처리: 편향(bias)을 위해 $X_0=1$ 특성을 추가하고, 타겟 y를 원-핫 인코딩으로 변환합니다.

In [2]:
# 편향(bias)을 위한 X0=1 특성 추가
X_with_bias = np.c_[np.ones([len(X), 1]), X]

# 타겟을 원-핫 인코딩으로 변환
n_classes = len(np.unique(y))
Y_one_hot = np.eye(n_classes)[y]

print("X_with_bias shape:", X_with_bias.shape)
print("Y_one_hot shape:", Y_one_hot.shape)
print("First 5 samples of X_with_bias:\n", X_with_bias[:5])
print("First 5 samples of Y_one_hot:\n", Y_one_hot[:5])

X_with_bias shape: (150, 5)
Y_one_hot shape: (150, 3)
First 5 samples of X_with_bias:
 [[1.  5.1 3.5 1.4 0.2]
 [1.  4.9 3.  1.4 0.2]
 [1.  4.7 3.2 1.3 0.2]
 [1.  4.6 3.1 1.5 0.2]
 [1.  5.  3.6 1.4 0.2]]
First 5 samples of Y_one_hot:
 [[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


훈련, 검증, 테스트 세트로 데이터를 분할.

In [3]:
test_ratio = 0.2
validation_ratio = 0.2
total_size = len(X_with_bias)

test_size = int(total_size * test_ratio)
validation_size = int(total_size * validation_ratio)
train_size = total_size - test_size - validation_size

# 데이터를 섞기 위해 인덱스를 무작위로 섞습니다.
rnd_idx = np.random.permutation(total_size)

X_train = X_with_bias[rnd_idx[:train_size]]
y_train = Y_one_hot[rnd_idx[:train_size]]
X_valid = X_with_bias[rnd_idx[train_size:-test_size]]
y_valid = Y_one_hot[rnd_idx[train_size:-test_size]]
X_test = X_with_bias[rnd_idx[-test_size:]]
y_test = Y_one_hot[rnd_idx[-test_size:]]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (90, 5)
y_train shape: (90, 3)
X_valid shape: (30, 5)
y_valid shape: (30, 3)
X_test shape: (30, 5)
y_test shape: (30, 3)


소프트맥스 함수, 크로스 엔트로피 비용 함수, 그레디언트 계산을 위한 함수들을 정의합니다.

In [4]:
def softmax(logits):
    exp_logits = np.exp(logits)
    sum_exp = np.sum(exp_logits, axis=1, keepdims=True)
    return exp_logits / sum_exp

def cross_entropy_loss(Y_one_hot, Y_proba):
    return -np.mean(np.sum(Y_one_hot * np.log(Y_proba + 1e-10), axis=1))

def calculate_gradients(X, Y_one_hot, Y_proba, m):
    return (1/m) * X.T @ (Y_proba - Y_one_hot)

이제 배치 경사 하강법과 조기 종료를 사용하여 소프트맥스 회귀 모델을 훈련합니다.

In [5]:
# 모델 파라미터 초기화
n_inputs = X_train.shape[1] # 특성 수 (편향 포함)
n_outputs = n_classes       # 클래스 수

Theta = np.random.randn(n_inputs, n_outputs)

# 하이퍼파라미터
eta = 0.1 # 학습률 (learning rate)
n_epochs = 5001 # 에포크 수

# 조기 종료(Early Stopping) 설정
best_loss = np.inf
best_Theta = None
epochs_without_improvement = 0
max_epochs_without_improvement = 50

for epoch in range(n_epochs):
    # 로짓 계산
    logits = X_train @ Theta
    # 소프트맥스 확률 계산
    Y_proba = softmax(logits)

    # 그레디언트 계산
    m = len(X_train)
    gradients = calculate_gradients(X_train, y_train, Y_proba, m)

    # 파라미터 업데이트
    Theta = Theta - eta * gradients

    # 검증 세트에서 손실 계산
    logits_valid = X_valid @ Theta
    Y_proba_valid = softmax(logits_valid)
    valid_loss = cross_entropy_loss(y_valid, Y_proba_valid)

    # 조기 종료 로직
    if valid_loss < best_loss:
        best_loss = valid_loss
        best_Theta = Theta
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement > max_epochs_without_improvement:
            print(f"Early stopping at epoch {epoch}!")
            break

    if epoch % 500 == 0:
        train_loss = cross_entropy_loss(y_train, Y_proba)
        print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

# 최적의 Theta 사용
Theta = best_Theta
print("\nTraining finished. Best validation loss:", best_loss)

Epoch 0, Train Loss: 6.4248, Valid Loss: 3.7548
Epoch 500, Train Loss: 0.1652, Valid Loss: 0.1434
Epoch 1000, Train Loss: 0.1170, Valid Loss: 0.1012
Epoch 1500, Train Loss: 0.0963, Valid Loss: 0.0835
Epoch 2000, Train Loss: 0.0843, Valid Loss: 0.0738
Epoch 2500, Train Loss: 0.0764, Valid Loss: 0.0676
Epoch 3000, Train Loss: 0.0706, Valid Loss: 0.0633
Epoch 3500, Train Loss: 0.0662, Valid Loss: 0.0602
Epoch 4000, Train Loss: 0.0627, Valid Loss: 0.0578
Epoch 4500, Train Loss: 0.0599, Valid Loss: 0.0559
Epoch 5000, Train Loss: 0.0575, Valid Loss: 0.0545

Training finished. Best validation loss: 0.05447377474793496


모델을 평가합니다. 테스트 세트에서 예측을 수행하고 정확도를 계산합니다.

In [6]:
# 테스트 세트에 대한 예측
logits_test = X_test @ Theta
Y_proba_test = softmax(logits_test)
y_pred = np.argmax(Y_proba_test, axis=1)

# 실제 클래스
y_test_actual = np.argmax(y_test, axis=1)

# 정확도 계산
accuracy = np.mean(y_pred == y_test_actual)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9333
